In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
project_path = "/content/drive/MyDrive/turkish_legal_rag"
processed_path = f"{project_path}/data/processed"
faiss_path = f"{project_path}/outputs/faiss"

In [3]:
!pip install sentence-transformers faiss-cpu rank-bm25 transformers accelerate bitsandbytes rouge-score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.8 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=6af032a6a0a90281881ec2c67eb995276dfacfbcfa514ce059bcf4562a20869e
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [4]:
import pandas as pd
import os

chunks_df = pd.read_csv(f"{processed_path}/retrieval_corpus.csv")
test_qa_df = pd.read_csv(f"{processed_path}/test_qa.csv")

print(chunks_df.shape)
print(test_qa_df.shape)

(3775, 5)
(1500, 2)


In [5]:
eval_df = test_qa_df.sample(20, random_state=42).reset_index(drop=True)
eval_df.head()

,question,answer
0,Anayasanın 101. Maddesiyle İlgili Tartışmalar ...,Cumhurbaşkanının seçilme şartlarının sınırları...
1,"Bir grup vatandaş, belirli bir etnik grubun di...","Anayasanın 10. Maddesi, herkesin kanun önünde ..."
2,"Bir grup akademisyen, yaşama hakkının sınırlan...","Evet, Anayasanın 17. Maddesi, herkesin yaşama ..."
3,Geçici madde 20 ne zaman eklendi?,20 mayıs 2016 tarihinde.
4,Videoda TCK 121 ihlali sabit değil mi?,Bu tür hususlar dosyalarında bilişimci bilirki...


In [6]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

CUDA available: True
Tesla T4


In [7]:
from huggingface_hub import login

login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [8]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

llm_model_name = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(llm_model_name)

model = AutoModelForCausalLM.from_pretrained(
    llm_model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

print("LLM loaded:", llm_model_name)

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

LLM loaded: mistralai/Mistral-7B-Instruct-v0.2


In [9]:
def build_rag_prompt(question, retrieved_contexts):
    context_text = "\n\n".join([
        f"[Context {i+1}]\n{ctx}"
        for i, ctx in enumerate(retrieved_contexts)
    ])

    prompt = f"""
You are a Turkish legal question answering assistant.
Answer the question only using the provided legal context.
If the answer is not found in the context, say that the answer cannot be found in the given context.
Give a clear and concise answer in Turkish.

Legal Context:
{context_text}

Question:
{question}

Answer:
"""
    return prompt.strip()

In [10]:
def generate_answer(prompt, max_new_tokens=180):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "Answer:" in generated_text:
        generated_text = generated_text.split("Answer:")[-1].strip()

    return generated_text

In [11]:
import faiss
import numpy as np
import pandas as pd
import re
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

In [12]:
chunks_df = pd.read_csv(f"{processed_path}/retrieval_corpus.csv")

embedding_model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
embedding_model = SentenceTransformer(embedding_model_name)

index = faiss.read_index(f"{faiss_path}/baseline_faiss.index")

print("Chunks:", chunks_df.shape)
print("FAISS vectors:", index.ntotal)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Chunks: (3775, 5)
FAISS vectors: 3775


In [13]:
def simple_turkish_tokenize(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zçğıöşü0-9\s]", " ", text)
    tokens = text.split()

    stopwords = {
        "ve", "veya", "ile", "de", "da", "bir", "bu", "şu", "o",
        "için", "gibi", "olarak", "olan", "kadar", "ise", "ancak",
        "çok", "daha", "en", "mi", "mı", "mu", "mü"
    }

    return [t for t in tokens if t not in stopwords and len(t) > 1]


tokenized_corpus = [
    simple_turkish_tokenize(text)
    for text in chunks_df["chunk_text"].astype(str).tolist()
]

bm25 = BM25Okapi(tokenized_corpus)

print("BM25 ready.")

BM25 ready.


In [14]:
def min_max_normalize(scores):
    scores = np.array(scores, dtype=np.float32)

    if scores.max() == scores.min():
        return np.zeros_like(scores)

    return (scores - scores.min()) / (scores.max() - scores.min())


def hybrid_retrieve_top_k(query, model, index, chunks_df, bm25, k=5, alpha=0.5):
    query_embedding = model.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(query_embedding)

    dense_scores, dense_indices = index.search(query_embedding, len(chunks_df))

    dense_scores = dense_scores[0]
    dense_indices = dense_indices[0]

    dense_score_map = {
        int(idx): float(score)
        for idx, score in zip(dense_indices, dense_scores)
    }

    dense_all_scores = np.array([
        dense_score_map.get(i, 0.0)
        for i in range(len(chunks_df))
    ])

    tokenized_query = simple_turkish_tokenize(query)
    bm25_scores = np.array(bm25.get_scores(tokenized_query))

    dense_norm = min_max_normalize(dense_all_scores)
    bm25_norm = min_max_normalize(bm25_scores)

    final_scores = alpha * dense_norm + (1 - alpha) * bm25_norm

    top_indices = np.argsort(final_scores)[::-1][:k]

    results = []

    for rank, idx in enumerate(top_indices, start=1):
        results.append({
            "rank": rank,
            "chunk_id": chunks_df.iloc[idx]["chunk_id"],
            "source": chunks_df.iloc[idx]["source"],
            "score": float(final_scores[idx]),
            "dense_score": float(dense_norm[idx]),
            "bm25_score": float(bm25_norm[idx]),
            "chunk_text": chunks_df.iloc[idx]["chunk_text"]
        })

    return results

In [15]:
test_results = hybrid_retrieve_top_k(
    "Egemenlik kime aittir?",
    embedding_model,
    index,
    chunks_df,
    bm25,
    k=3,
    alpha=0.5
)

for r in test_results:
    print(r["rank"], r["chunk_id"], r["score"])
    print(r["chunk_text"][:300])
    print("-" * 80)

1 chunk_000269 0.9583597779273987
Madde 6 – Egemenlik, kayıtsız şartsız Milletindir. Türk Milleti, egemenliğini, Anayasanın koyduğu esaslara göre, yetkili organları eliyle kullanır.
--------------------------------------------------------------------------------
2 chunk_003519 0.898059606552124
DÖRDÜNCÜ KISIM
Millete ve Devlete Karşı Suçlar ve Son Hükümler ÜÇÜNCÜ BÖLÜM
Devletin Egemenlik Alametlerine ve Organlarının Saygınlığına Karşı Suçlar
--------------------------------------------------------------------------------
3 chunk_002321 0.804427444934845
Madde 960- Ortaklık genel kurulunda rehinli pay senetlerini temsil etmek yetkisi, rehin 
alacaklısına değil, pay sahibine aittir.
II I. Yönetim ve ödeme
--------------------------------------------------------------------------------


In [16]:
sample_row = eval_df.iloc[0]

question = sample_row["question"]
expected_answer = sample_row["answer"]

retrieved = hybrid_retrieve_top_k(
    question,
    embedding_model,
    index,
    chunks_df,
    bm25,
    k=3,
    alpha=0.5
)

contexts = [item["chunk_text"] for item in retrieved]
prompt = build_rag_prompt(question, contexts)

generated_answer = generate_answer(prompt)

print("QUESTION:")
print(question)

print("\nEXPECTED ANSWER:")
print(expected_answer)

print("\nGENERATED ANSWER:")
print(generated_answer)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


QUESTION:
Anayasanın 101. Maddesiyle İlgili Tartışmalar Nelerdir?

EXPECTED ANSWER:
Cumhurbaşkanının seçilme şartlarının sınırları ve uygulanması üzerine tartışmalar olabilir.

GENERATED ANSWER:
Anayasanın 101. Madde'si, vakıflar hakkında söz eder. Bu madde, vakıfların yeterli mal ve haklarının vakfedilebilir olduğunu ve bunların anlaşılabilen her türlü geliri veya ekonomik değeri olan hakların vakfedilebilir olduğunu söyler. Anayasanın temel ilkelerine, hukuka, ahlâka, millî birliğe ve millî menfaatlara aykırı veya belli bir ırk ya da cemaat mensupl


In [17]:
results = hybrid_retrieve_top_k(
    question,
    embedding_model,
    index,
    chunks_df,
    bm25,
    k=5,
    alpha=0.5
)

for r in results:
    print("=" * 80)
    print(r["chunk_id"], r["score"])
    print(r["chunk_text"][:500])

chunk_002446 0.9372999668121338
Madde 101- Vakıflar, gerçek veya tüzel kişilerin yeterli mal ve hakları belirli ve sürekli 
bir amaca özgülemeleriyle oluşan tüzel kişiliğe sahip mal topluluklarıdır. 
Bir malvarlığının bütünü veya gerçekleşmiş ya da gerçekleşeceği anlaşılan her türlü geliri 
veya ekonomik değeri olan haklar vakfedilebilir.
(İptal üçüncü fıkra: Anayasa Mahkemesi’nin 17/4/2008 tarihli ve E.: 2005/14, K.:
2008/92 sayılı Kararı ile.)
Cumhuriyetin Anayasa ile belirlenen niteliklerine ve Anayasanın temel ilkele
chunk_000355 0.9126801490783691
Madde 176 – Anayasanın dayandığı temel görüş ve ilkeleri belirten başlangıç kısmı, Anayasa metnine dahildir.
chunk_000210 0.8682551383972168
Madde 158 – Uyuşmazlık Mahkemesi adli ve idari yargı mercileri arasındaki görev ve hüküm uyuşmazlıklarını kesin olarak çözümlemeye yetkilidir. [101]
chunk_000362 0.8658927083015442
e) Anayasanın halkoylaması sonucu kabulünün ilanıyle birlikte yürürlüğe girecek hükümleri ve mevcut ve kurulacak kurum,

In [18]:
results = hybrid_retrieve_top_k(
    question,
    embedding_model,
    index,
    chunks_df,
    bm25,
    k=10,
    alpha=0.5
)

for r in results:
    print("=" * 80)
    print(r["rank"], r["chunk_id"], r["score"], r["source"])
    print(r["chunk_text"][:500])

1 chunk_002446 0.9372999668121338 Türk Medeni Kanunu
Madde 101- Vakıflar, gerçek veya tüzel kişilerin yeterli mal ve hakları belirli ve sürekli 
bir amaca özgülemeleriyle oluşan tüzel kişiliğe sahip mal topluluklarıdır. 
Bir malvarlığının bütünü veya gerçekleşmiş ya da gerçekleşeceği anlaşılan her türlü geliri 
veya ekonomik değeri olan haklar vakfedilebilir.
(İptal üçüncü fıkra: Anayasa Mahkemesi’nin 17/4/2008 tarihli ve E.: 2005/14, K.:
2008/92 sayılı Kararı ile.)
Cumhuriyetin Anayasa ile belirlenen niteliklerine ve Anayasanın temel ilkele
2 chunk_000355 0.9126801490783691 Türkiye Cumhuriyeti Anayasası
Madde 176 – Anayasanın dayandığı temel görüş ve ilkeleri belirten başlangıç kısmı, Anayasa metnine dahildir.
3 chunk_000210 0.8682551383972168 Türkiye Cumhuriyeti Anayasası
Madde 158 – Uyuşmazlık Mahkemesi adli ve idari yargı mercileri arasındaki görev ve hüküm uyuşmazlıklarını kesin olarak çözümlemeye yetkilidir. [101]
4 chunk_000362 0.8658927083015442 Türkiye Cumhuriyeti Anayasası
e)

In [19]:
def detect_source_filter(query):
    q = str(query).lower()

    if "anayasa" in q or "anayasanın" in q:
        return "Türkiye Cumhuriyeti Anayasası"

    return None

In [20]:
def hybrid_retrieve_top_k_filtered(query, model, index, chunks_df, bm25, k=5, alpha=0.5):
    source_filter = detect_source_filter(query)

    candidate_df = chunks_df.copy()

    if source_filter is not None:
        candidate_df = candidate_df[
            candidate_df["source"].astype(str).str.lower() == source_filter.lower()
        ].reset_index(drop=True)

    if len(candidate_df) == 0:
        candidate_df = chunks_df.copy()

    # Bu basit çözüm: filtreli corpus için temporary dense index kuruyoruz
    candidate_texts = candidate_df["chunk_text"].astype(str).tolist()

    candidate_embeddings = embedding_model.encode(
        candidate_texts,
        convert_to_numpy=True,
        show_progress_bar=False
    ).astype("float32")

    faiss.normalize_L2(candidate_embeddings)

    temp_index = faiss.IndexFlatIP(candidate_embeddings.shape[1])
    temp_index.add(candidate_embeddings)

    tokenized_candidate_corpus = [
        simple_turkish_tokenize(text)
        for text in candidate_texts
    ]

    temp_bm25 = BM25Okapi(tokenized_candidate_corpus)

    return hybrid_retrieve_top_k(
        query,
        model,
        temp_index,
        candidate_df,
        temp_bm25,
        k=k,
        alpha=alpha
    )

In [21]:
filtered_results = hybrid_retrieve_top_k_filtered(
    question,
    embedding_model,
    index,
    chunks_df,
    bm25,
    k=5,
    alpha=0.5
)

for r in filtered_results:
    print("=" * 80)
    print(r["rank"], r["chunk_id"], r["score"], r["source"])
    print(r["chunk_text"][:600])

1 chunk_000210 0.8626564741134644 Türkiye Cumhuriyeti Anayasası
Madde 158 – Uyuşmazlık Mahkemesi adli ve idari yargı mercileri arasındaki görev ve hüküm uyuşmazlıklarını kesin olarak çözümlemeye yetkilidir. [101]
2 chunk_000355 0.7758653163909912 Türkiye Cumhuriyeti Anayasası
Madde 176 – Anayasanın dayandığı temel görüş ve ilkeleri belirten başlangıç kısmı, Anayasa metnine dahildir.
3 chunk_000537 0.7254406213760376 Türkiye Cumhuriyeti Anayasası
Cumhurbaşkanlığı seçiminde birinci oylamada gerekli çoğunluğun sağlanamaması halinde 101 inci maddedeki usule göre ikinci oylama yapılır. D. Seçimlerin geriye bırakılması ve ara seçimler [34]
4 chunk_000362 0.7193549275398254 Türkiye Cumhuriyeti Anayasası
e) Anayasanın halkoylaması sonucu kabulünün ilanıyle birlikte yürürlüğe girecek hükümleri ve mevcut ve kurulacak kurum, kuruluş ve kurullar için yeniden kanun yapılması veya mevcut kanunlarda değişiklik yapılması gerekiyorsa bunlara ilişkin işlemler mevcut kanunların Anayasaya aykırı olmayan h

In [22]:
sample_questions = [
    "Egemenlik kime aittir?",
    "Türkiye Cumhuriyetinin yönetim şekli nedir?",
    "Cumhurbaşkanının görev süresi kaç yıldır?",
    "Bir kimse en fazla kaç defa Cumhurbaşkanı seçilebilir?"
]

for q in sample_questions:
    print("=" * 100)
    print("QUESTION:", q)

    results = hybrid_retrieve_top_k_filtered(
        q,
        embedding_model,
        index,
        chunks_df,
        bm25,
        k=5,
        alpha=0.5
    )

    for r in results[:3]:
        print(r["rank"], r["chunk_id"], r["score"], r["source"])
        print(r["chunk_text"][:500])
        print("-" * 80)

QUESTION: Egemenlik kime aittir?
1 chunk_000269 0.9583597183227539 Türkiye Cumhuriyeti Anayasası
Madde 6 – Egemenlik, kayıtsız şartsız Milletindir. Türk Milleti, egemenliğini, Anayasanın koyduğu esaslara göre, yetkili organları eliyle kullanır.
--------------------------------------------------------------------------------
2 chunk_003519 0.8980596661567688 Türk Ceza Kanunu
DÖRDÜNCÜ KISIM
Millete ve Devlete Karşı Suçlar ve Son Hükümler ÜÇÜNCÜ BÖLÜM
Devletin Egemenlik Alametlerine ve Organlarının Saygınlığına Karşı Suçlar
--------------------------------------------------------------------------------
3 chunk_002321 0.8044273853302002 Türk Medeni Kanunu
Madde 960- Ortaklık genel kurulunda rehinli pay senetlerini temsil etmek yetkisi, rehin 
alacaklısına değil, pay sahibine aittir.
II I. Yönetim ve ödeme
--------------------------------------------------------------------------------
QUESTION: Türkiye Cumhuriyetinin yönetim şekli nedir?
1 chunk_000000 0.9103504419326782 Türkiye Cumhuriye

In [23]:
for q in sample_questions:
    print("=" * 100)
    print("QUESTION:", q)

    results = hybrid_retrieve_top_k_filtered(
        q,
        embedding_model,
        index,
        chunks_df,
        bm25,
        k=5,
        alpha=0.5
    )

    contexts = [r["chunk_text"] for r in results[:3]]

    prompt = build_rag_prompt(q, contexts)

    answer = generate_answer(prompt)

    print("\nGENERATED ANSWER:")
    print(answer)

QUESTION: Egemenlik kime aittir?

GENERATED ANSWER:
Egemenlik, Türk Milleti tarafından Anayasanın koyduğu esaslara göre, yetkili organları eliyle kullanılır (Context 1).

Therefore, the answer to the question is: "Egemenlik, Türk Milleti'ne aitdir." (Egemenlik belongs to the Turkish Nation.)
QUESTION: Türkiye Cumhuriyetinin yönetim şekli nedir?

GENERATED ANSWER:
Türkiye Cumhuriyetinin yönetim şekli, Cumhuriyet Sistemi (Cumhuriyet Cumhuriyetidir, önemli olarak Atatürk'un milliyetçilik anlayışına dayalı bir yönetim şekli olduğunu belirten Anayasasının kısaca, Türkiye Cumhuriyetinin ebedi varlığını ve Yüce Türk Devletinin bölünmez bütünlüğünü belirleyen bir anayasadır. Bu anayasanın kısaca, Türki
QUESTION: Cumhurbaşkanının görev süresi kaç yıldır?

GENERATED ANSWER:
Cumhurbaşkanının görev süresi beş yıldır. (Context 1)
QUESTION: Bir kimse en fazla kaç defa Cumhurbaşkanı seçilebilir?

GENERATED ANSWER:
En fazla üç defa Cumhurbaşkanı seçilebilir. Context 1'de belirtilen kurallara göre, bir

In [24]:
base_eval_questions = [
    {
        "question": "Egemenlik kime aittir?",
        "expected_answer": "Egemenlik kayıtsız şartsız Milletindir."
    },
    {
        "question": "Türkiye Cumhuriyetinin yönetim şekli nedir?",
        "expected_answer": "Türkiye Devleti bir Cumhuriyettir."
    },
    {
        "question": "Cumhurbaşkanının görev süresi kaç yıldır?",
        "expected_answer": "Cumhurbaşkanının görev süresi beş yıldır."
    },
    {
        "question": "Bir kimse en fazla kaç defa Cumhurbaşkanı seçilebilir?",
        "expected_answer": "Bir kimse en fazla iki defa Cumhurbaşkanı seçilebilir."
    }
]

base_eval_df = pd.DataFrame(base_eval_questions)
base_eval_df

,question,expected_answer
0,Egemenlik kime aittir?,Egemenlik kayıtsız şartsız Milletindir.
1,Türkiye Cumhuriyetinin yönetim şekli nedir?,Türkiye Devleti bir Cumhuriyettir.
2,Cumhurbaşkanının görev süresi kaç yıldır?,Cumhurbaşkanının görev süresi beş yıldır.
3,Bir kimse en fazla kaç defa Cumhurbaşkanı seçi...,Bir kimse en fazla iki defa Cumhurbaşkanı seçi...


In [25]:
base_results = []

for _, row in base_eval_df.iterrows():
    question = row["question"]
    expected_answer = row["expected_answer"]

    retrieved = hybrid_retrieve_top_k_filtered(
        question,
        embedding_model,
        index,
        chunks_df,
        bm25,
        k=5,
        alpha=0.5
    )

    contexts = [r["chunk_text"] for r in retrieved[:3]]
    prompt = build_rag_prompt(question, contexts)
    generated_answer = generate_answer(prompt)

    base_results.append({
        "question": question,
        "expected_answer": expected_answer,
        "generated_answer": generated_answer,
        "top1_chunk_id": retrieved[0]["chunk_id"],
        "top1_context": retrieved[0]["chunk_text"]
    })

base_rag_results_df = pd.DataFrame(base_results)
base_rag_results_df

,question,expected_answer,generated_answer,top1_chunk_id,top1_context
0,Egemenlik kime aittir?,Egemenlik kayıtsız şartsız Milletindir.,"Egemenlik, Türk Milleti tarafından Anayasanın ...",chunk_000269,"Madde 6 – Egemenlik, kayıtsız şartsız Milletin..."
1,Türkiye Cumhuriyetinin yönetim şekli nedir?,Türkiye Devleti bir Cumhuriyettir.,"Türkiye Cumhuriyetinin yönetim şekli, Cumhuriy...",chunk_000000,Türk Vatanı ve Milletinin ebedi varlığını ve Y...
2,Cumhurbaşkanının görev süresi kaç yıldır?,Cumhurbaşkanının görev süresi beş yıldır.,Cumhurbaşkanının görev süresi beş yıldır. (Con...,chunk_000043,"Cumhurbaşkanı, kırk yaşını doldurmuş, yükseköğ..."
3,Bir kimse en fazla kaç defa Cumhurbaşkanı seçi...,Bir kimse en fazla iki defa Cumhurbaşkanı seçi...,En fazla üç defa Cumhurbaşkanı seçilebilir. Co...,chunk_000043,"Cumhurbaşkanı, kırk yaşını doldurmuş, yükseköğ..."


In [26]:
for i, row in base_rag_results_df.iterrows():
    print("=" * 100)
    print("Index:", i)
    print("QUESTION:")
    print(row["question"])
    print("\nEXPECTED:")
    print(row["expected_answer"])
    print("\nGENERATED:")
    print(row["generated_answer"])

Index: 0
QUESTION:
Egemenlik kime aittir?

EXPECTED:
Egemenlik kayıtsız şartsız Milletindir.

GENERATED:
Egemenlik, Türk Milleti tarafından Anayasanın koyduğu esaslara göre, yetkili organları eliyle kullanılır (Context 1).

Therefore, the answer to the question is: "Egemenlik, Türk Milleti'ne aitdir." (Egemenlik belongs to the Turkish Nation.)
Index: 1
QUESTION:
Türkiye Cumhuriyetinin yönetim şekli nedir?

EXPECTED:
Türkiye Devleti bir Cumhuriyettir.

GENERATED:
Türkiye Cumhuriyetinin yönetim şekli, Cumhuriyet Sistemi (Cumhuriyet Cumhuriyetidir, önemli olarak Atatürk'un milliyetçilik anlayışına dayalı bir yönetim şekli olduğunu belirten Anayasasının kısaca, Türkiye Cumhuriyetinin ebedi varlığını ve Yüce Türk Devletinin bölünmez bütünlüğünü belirleyen bir anayasadır. Bu anayasanın kısaca, Türki
Index: 2
QUESTION:
Cumhurbaşkanının görev süresi kaç yıldır?

EXPECTED:
Cumhurbaşkanının görev süresi beş yıldır.

GENERATED:
Cumhurbaşkanının görev süresi beş yıldır. (Context 1)
Index: 3
QUESTI

In [27]:
manual_scores = [1.0, 0.5, 1.0, 0.0]

base_rag_results_df["manual_score"] = manual_scores

base_score = base_rag_results_df["manual_score"].mean()

print("Base RAG Manual Accuracy:", base_score)
base_rag_results_df

Base RAG Manual Accuracy: 0.625


,question,expected_answer,generated_answer,top1_chunk_id,top1_context,manual_score
0,Egemenlik kime aittir?,Egemenlik kayıtsız şartsız Milletindir.,"Egemenlik, Türk Milleti tarafından Anayasanın ...",chunk_000269,"Madde 6 – Egemenlik, kayıtsız şartsız Milletin...",1.0
1,Türkiye Cumhuriyetinin yönetim şekli nedir?,Türkiye Devleti bir Cumhuriyettir.,"Türkiye Cumhuriyetinin yönetim şekli, Cumhuriy...",chunk_000000,Türk Vatanı ve Milletinin ebedi varlığını ve Y...,0.5
2,Cumhurbaşkanının görev süresi kaç yıldır?,Cumhurbaşkanının görev süresi beş yıldır.,Cumhurbaşkanının görev süresi beş yıldır. (Con...,chunk_000043,"Cumhurbaşkanı, kırk yaşını doldurmuş, yükseköğ...",1.0
3,Bir kimse en fazla kaç defa Cumhurbaşkanı seçi...,Bir kimse en fazla iki defa Cumhurbaşkanı seçi...,En fazla üç defa Cumhurbaşkanı seçilebilir. Co...,chunk_000043,"Cumhurbaşkanı, kırk yaşını doldurmuş, yükseköğ...",0.0


In [28]:
base_rag_results_df.to_csv(
    f"{project_path}/outputs/metrics/base_rag_generation_results.csv",
    index=False,
    encoding="utf-8-sig"
)

pd.DataFrame([{
    "method": "Base RAG - Hybrid Retrieval + Base LLM",
    "manual_accuracy": base_score
}]).to_csv(
    f"{project_path}/outputs/metrics/base_rag_manual_score.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Base RAG results saved.")

Base RAG results saved.


## Experiment 2: Prompt-Improved RAG

In [29]:
def build_strict_rag_prompt(question, retrieved_contexts):
    context_text = "\n\n".join([
        f"[Context {i+1}]\n{ctx}"
        for i, ctx in enumerate(retrieved_contexts)
    ])

    prompt = f"""
Sen Türk hukuk metinleri için çalışan dikkatli bir soru-cevap asistanısın.

Kurallar:
- Cevabı SADECE verilen bağlama göre ver.
- Bağlamda açıkça yazmayan çıkarımları yapma.
- Birden fazla bağlam çelişirse en doğrudan cevap veren bağlamı kullan.
- Cevap Türkçe olmalı.
- Cevap kısa ve net olmalı.
- İngilizce açıklama, "Therefore", "Context 1" gibi ifadeler yazma.

Bağlam:
{context_text}

Soru:
{question}

Kısa cevap:
"""
    return prompt.strip()

In [30]:
improved_results = []

for _, row in base_eval_df.iterrows():
    question = row["question"]
    expected_answer = row["expected_answer"]

    retrieved = hybrid_retrieve_top_k_filtered(
        question,
        embedding_model,
        index,
        chunks_df,
        bm25,
        k=5,
        alpha=0.5
    )

    contexts = [r["chunk_text"] for r in retrieved[:3]]

    prompt = build_strict_rag_prompt(question, contexts)

    generated_answer = generate_answer(prompt)

    improved_results.append({
        "question": question,
        "expected_answer": expected_answer,
        "generated_answer": generated_answer
    })

improved_rag_results_df = pd.DataFrame(improved_results)
improved_rag_results_df

,question,expected_answer,generated_answer
0,Egemenlik kime aittir?,Egemenlik kayıtsız şartsız Milletindir.,Sen Türk hukuk metinleri için çalışan dikkatli...
1,Türkiye Cumhuriyetinin yönetim şekli nedir?,Türkiye Devleti bir Cumhuriyettir.,Sen Türk hukuk metinleri için çalışan dikkatli...
2,Cumhurbaşkanının görev süresi kaç yıldır?,Cumhurbaşkanının görev süresi beş yıldır.,Sen Türk hukuk metinleri için çalışan dikkatli...
3,Bir kimse en fazla kaç defa Cumhurbaşkanı seçi...,Bir kimse en fazla iki defa Cumhurbaşkanı seçi...,Sen Türk hukuk metinleri için çalışan dikkatli...


In [31]:
for i, row in improved_rag_results_df.iterrows():
    print("=" * 100)
    print("QUESTION:")
    print(row["question"])
    print("\nEXPECTED:")
    print(row["expected_answer"])
    print("\nGENERATED:")
    print(row["generated_answer"])

QUESTION:
Egemenlik kime aittir?

EXPECTED:
Egemenlik kayıtsız şartsız Milletindir.

GENERATED:
Sen Türk hukuk metinleri için çalışan dikkatli bir soru-cevap asistanısın.

Kurallar:
- Cevabı SADECE verilen bağlama göre ver.
- Bağlamda açıkça yazmayan çıkarımları yapma.
- Birden fazla bağlam çelişirse en doğrudan cevap veren bağlamı kullan.
- Cevap Türkçe olmalı.
- Cevap kısa ve net olmalı.
- İngilizce açıklama, "Therefore", "Context 1" gibi ifadeler yazma.

Bağlam:
[Context 1]
Madde 6 – Egemenlik, kayıtsız şartsız Milletindir. Türk Milleti, egemenliğini, Anayasanın koyduğu esaslara göre, yetkili organları eliyle kullanır.

[Context 2]
DÖRDÜNCÜ KISIM
Millete ve Devlete Karşı Suçlar ve Son Hükümler ÜÇÜNCÜ BÖLÜM
Devletin Egemenlik Alametlerine ve Organlarının Saygınlığına Karşı Suçlar

[Context 3]
Madde 960- Ortaklık genel kurulunda rehinli pay senetlerini temsil etmek yetkisi, rehin 
alacaklısına değil, pay sahibine aittir.
II I. Yönetim ve ödeme

Soru:
Egemenlik kime aittir?

Kısa cevap

In [32]:
manual_scores_improved = [1.0, 0.0, 1.0, 1.0]

improved_rag_results_df["manual_score"] = manual_scores_improved

improved_score = improved_rag_results_df["manual_score"].mean()

print("Improved Prompt RAG Accuracy:", improved_score)
improved_rag_results_df

Improved Prompt RAG Accuracy: 0.75


,question,expected_answer,generated_answer,manual_score
0,Egemenlik kime aittir?,Egemenlik kayıtsız şartsız Milletindir.,Sen Türk hukuk metinleri için çalışan dikkatli...,1.0
1,Türkiye Cumhuriyetinin yönetim şekli nedir?,Türkiye Devleti bir Cumhuriyettir.,Sen Türk hukuk metinleri için çalışan dikkatli...,0.0
2,Cumhurbaşkanının görev süresi kaç yıldır?,Cumhurbaşkanının görev süresi beş yıldır.,Sen Türk hukuk metinleri için çalışan dikkatli...,1.0
3,Bir kimse en fazla kaç defa Cumhurbaşkanı seçi...,Bir kimse en fazla iki defa Cumhurbaşkanı seçi...,Sen Türk hukuk metinleri için çalışan dikkatli...,1.0


In [33]:
def clean_generated_answer(text):
    if "Kısa cevap:" in text:
        text = text.split("Kısa cevap:")[-1].strip()

    if "Detaylı cevap:" in text:
        text = text.split("Detaylı cevap:")[0].strip()

    return text.strip()

In [34]:
improved_rag_results_df.to_csv(
    f"{project_path}/outputs/metrics/prompt_improved_rag_generation_results.csv",
    index=False,
    encoding="utf-8-sig"
)

pd.DataFrame([{
    "method": "Prompt Improved RAG - Hybrid Retrieval + Base LLM",
    "manual_accuracy": improved_score
}]).to_csv(
    f"{project_path}/outputs/metrics/prompt_improved_rag_manual_score.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Prompt improved RAG results saved.")

Prompt improved RAG results saved.


In [35]:
rag_comparison_df = pd.DataFrame([
    {
        "method": "Base RAG",
        "manual_accuracy": base_score
    },
    {
        "method": "Prompt Improved RAG",
        "manual_accuracy": improved_score
    }
])

rag_comparison_df

,method,manual_accuracy
0,Base RAG,0.625
1,Prompt Improved RAG,0.750


## Test Set Evaluation

In [36]:
test_qa_df = pd.read_csv(f"{project_path}/data/processed/test_qa.csv")

test_eval_df = test_qa_df.sample(n=20, random_state=42).reset_index(drop=True)

print(test_eval_df.shape)
test_eval_df.head()

(20, 2)


,question,answer
0,Anayasanın 101. Maddesiyle İlgili Tartışmalar ...,Cumhurbaşkanının seçilme şartlarının sınırları...
1,"Bir grup vatandaş, belirli bir etnik grubun di...","Anayasanın 10. Maddesi, herkesin kanun önünde ..."
2,"Bir grup akademisyen, yaşama hakkının sınırlan...","Evet, Anayasanın 17. Maddesi, herkesin yaşama ..."
3,Geçici madde 20 ne zaman eklendi?,20 mayıs 2016 tarihinde.
4,Videoda TCK 121 ihlali sabit değil mi?,Bu tür hususlar dosyalarında bilişimci bilirki...


In [37]:
base_test_results = []

for _, row in test_eval_df.iterrows():
    question = row["question"]
    expected_answer = row["answer"]

    retrieved = hybrid_retrieve_top_k_filtered(
        question,
        embedding_model,
        index,
        chunks_df,
        bm25,
        k=5,
        alpha=0.5
    )

    contexts = [r["chunk_text"] for r in retrieved[:3]]
    prompt = build_rag_prompt(question, contexts)
    generated_answer = generate_answer(prompt)

    base_test_results.append({
        "question": question,
        "expected_answer": expected_answer,
        "generated_answer": generated_answer,
        "top1_chunk_id": retrieved[0]["chunk_id"],
        "top1_context": retrieved[0]["chunk_text"]
    })

base_test_results_df = pd.DataFrame(base_test_results)
base_test_results_df.head()

,question,expected_answer,generated_answer,top1_chunk_id,top1_context
0,Anayasanın 101. Maddesiyle İlgili Tartışmalar ...,Cumhurbaşkanının seçilme şartlarının sınırları...,"Anayasanın 101. Madde'si, Uyuşmazlık Mahkemesi...",chunk_000210,Madde 158 – Uyuşmazlık Mahkemesi adli ve idari...
1,"Bir grup vatandaş, belirli bir etnik grubun di...","Anayasanın 10. Maddesi, herkesin kanun önünde ...",Bu durum Anayasanın 150. Madde ile ilgili olab...,chunk_000188,"Madde 150 – Kanunların, Cumhurbaşkanlığı karar..."
2,"Bir grup akademisyen, yaşama hakkının sınırlan...","Evet, Anayasanın 17. Maddesi, herkesin yaşama ...",Anayasanın 17. Maddesine aykırı değildir. Bu m...,chunk_000373,"Madde 17 – Herkes, yaşama, maddi ve manevi var..."
3,Geçici madde 20 ne zaman eklendi?,20 mayıs 2016 tarihinde.,"Geçici madde 20, Context 1'de belirtilen tarih...",chunk_000600,Madde 20- Açıklanması veya zamanından önce açı...
4,Videoda TCK 121 ihlali sabit değil mi?,Bu tür hususlar dosyalarında bilişimci bilirki...,"Gözlenen videoda, TCK 121 ihlali hakkında beli...",chunk_000101,Madde 121 – (Mülga: 21/1/2017-6771/16 md.) B. ...


In [38]:
strict_test_results = []

for _, row in test_eval_df.iterrows():
    question = row["question"]
    expected_answer = row["answer"]

    retrieved = hybrid_retrieve_top_k_filtered(
        question,
        embedding_model,
        index,
        chunks_df,
        bm25,
        k=5,
        alpha=0.5
    )

    contexts = [r["chunk_text"] for r in retrieved[:3]]
    prompt = build_strict_rag_prompt(question, contexts)
    generated_answer = generate_answer(prompt)

    strict_test_results.append({
        "question": question,
        "expected_answer": expected_answer,
        "generated_answer": generated_answer,
        "top1_chunk_id": retrieved[0]["chunk_id"],
        "top1_context": retrieved[0]["chunk_text"]
    })

strict_test_results_df = pd.DataFrame(strict_test_results)
strict_test_results_df.head()

,question,expected_answer,generated_answer,top1_chunk_id,top1_context
0,Anayasanın 101. Maddesiyle İlgili Tartışmalar ...,Cumhurbaşkanının seçilme şartlarının sınırları...,Sen Türk hukuk metinleri için çalışan dikkatli...,chunk_000210,Madde 158 – Uyuşmazlık Mahkemesi adli ve idari...
1,"Bir grup vatandaş, belirli bir etnik grubun di...","Anayasanın 10. Maddesi, herkesin kanun önünde ...",Sen Türk hukuk metinleri için çalışan dikkatli...,chunk_000188,"Madde 150 – Kanunların, Cumhurbaşkanlığı karar..."
2,"Bir grup akademisyen, yaşama hakkının sınırlan...","Evet, Anayasanın 17. Maddesi, herkesin yaşama ...",Sen Türk hukuk metinleri için çalışan dikkatli...,chunk_000373,"Madde 17 – Herkes, yaşama, maddi ve manevi var..."
3,Geçici madde 20 ne zaman eklendi?,20 mayıs 2016 tarihinde.,Sen Türk hukuk metinleri için çalışan dikkatli...,chunk_000600,Madde 20- Açıklanması veya zamanından önce açı...
4,Videoda TCK 121 ihlali sabit değil mi?,Bu tür hususlar dosyalarında bilişimci bilirki...,Sen Türk hukuk metinleri için çalışan dikkatli...,chunk_000101,Madde 121 – (Mülga: 21/1/2017-6771/16 md.) B. ...


In [39]:
base_test_results_df.to_csv(
    f"{project_path}/outputs/metrics/base_rag_testset_generation_results.csv",
    index=False,
    encoding="utf-8-sig"
)

strict_test_results_df.to_csv(
    f"{project_path}/outputs/metrics/strict_prompt_rag_testset_generation_results.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Test set generation results saved.")

Test set generation results saved.


In [40]:
for i, row in strict_test_results_df.iterrows():
    print("=" * 120)
    print("INDEX:", i)
    print("QUESTION:")
    print(row["question"])
    print("\nEXPECTED:")
    print(row["expected_answer"])
    print("\nGENERATED:")
    print(row["generated_answer"][:1000])

INDEX: 0
QUESTION:
Anayasanın 101. Maddesiyle İlgili Tartışmalar Nelerdir?

EXPECTED:
Cumhurbaşkanının seçilme şartlarının sınırları ve uygulanması üzerine tartışmalar olabilir.

GENERATED:
Sen Türk hukuk metinleri için çalışan dikkatli bir soru-cevap asistanısın.

Kurallar:
- Cevabı SADECE verilen bağlama göre ver.
- Bağlamda açıkça yazmayan çıkarımları yapma.
- Birden fazla bağlam çelişirse en doğrudan cevap veren bağlamı kullan.
- Cevap Türkçe olmalı.
- Cevap kısa ve net olmalı.
- İngilizce açıklama, "Therefore", "Context 1" gibi ifadeler yazma.

Bağlam:
[Context 1]
Madde 158 – Uyuşmazlık Mahkemesi adli ve idari yargı mercileri arasındaki görev ve hüküm uyuşmazlıklarını kesin olarak çözümlemeye yetkilidir. [101]

[Context 2]
Madde 176 – Anayasanın dayandığı temel görüş ve ilkeleri belirten başlangıç kısmı, Anayasa metnine dahildir.

[Context 3]
Cumhurbaşkanlığı seçiminde birinci oylamada gerekli çoğunluğun sağlanamaması halinde 101 inci maddedeki usule göre ikinci oylama yapılır. D.

In [41]:
def clean_generated_answer(text):
    text = str(text)

    if "Kısa cevap:" in text:
        text = text.split("Kısa cevap:")[-1].strip()

    if "Detaylı cevap:" in text:
        text = text.split("Detaylı cevap:")[0].strip()

    return text.strip()

strict_test_results_df["clean_generated_answer"] = strict_test_results_df["generated_answer"].apply(clean_generated_answer)

In [42]:
for i, row in strict_test_results_df.iterrows():
    print("=" * 120)
    print("INDEX:", i)
    print("QUESTION:")
    print(row["question"])
    print("\nEXPECTED:")
    print(row["expected_answer"])
    print("\nCLEAN GENERATED:")
    print(row["clean_generated_answer"][:1000])

INDEX: 0
QUESTION:
Anayasanın 101. Maddesiyle İlgili Tartışmalar Nelerdir?

EXPECTED:
Cumhurbaşkanının seçilme şartlarının sınırları ve uygulanması üzerine tartışmalar olabilir.

CLEAN GENERATED:
Anayasanın 101. Maddesiyle ilgili tartışmalar, Uyuşmazlık Mahkemesi adli ve idari yargı mercileri arasındaki görev ve hüküm uyuşmazlıklarının çözülmesi hakkındadır.
INDEX: 1
QUESTION:
Bir grup vatandaş, belirli bir etnik grubun diğerlerinden daha fazla hakka sahip olması için imza kampanyası başlatmıştır. Bu durum Anayasanın 10. Maddesi ile nasıl çelişir?

EXPECTED:
Anayasanın 10. Maddesi, herkesin kanun önünde eşit olduğunu belirtir. Bu tür bir imza kampanyası Anayasa'ya aykırıdır.

CLEAN GENERATED:
Anayasanın 10. Maddesi, Türkiye'nin bütün vatandaşlarının halkoyluğunu temsil etmesine dayalı olarak, Cumhurbaşkanına, Türkiye Büyük Millet Meclisinde en fazla üyeye sahip iki siyasi parti grubuna ve üye tamsayısının en az beşte biri tutarındaki üyelere ait hakkı tanımlar. Bu durumda, etnik grubun

In [43]:
strict_test_results_df["is_valid_sample"] = True

# Dataset hatalı/uyumsuz sample
strict_test_results_df.loc[19, "is_valid_sample"] = False

manual_scores_strict = [
    0.0,  # 0 - yanlış context / yanlış cevap
    0.0,  # 1 - yanlış
    0.5,  # 2 - kısmen, yaşama hakkını yakalamış ama aykırılık yorumu ters
    0.0,  # 3 - yanlış tarih
    0.0,  # 4 - yanlış / alakasız
    0.0,  # 5 - yanlış
    0.0,  # 6 - yanlış/eksik
    1.0,  # 7 - yedi gün doğru
    0.0,  # 8 - yanlış
    0.0,  # 9 - yanlış
    0.5,  # 10 - kısmen doğru
    0.0,  # 11 - yanlış
    0.0,  # 12 - yanlış
    1.0,  # 13 - doğru
    0.5,  # 14 - kısmen doğru
    0.5,  # 15 - kısmen doğru
    0.5,  # 16 - kısmen doğru
    0.0,  # 17 - yanlış / cevap yok
    0.5,  # 18 - kısmen doğru
    0.0   # 19 - invalid olacak, skora katılmayacak
]

strict_test_results_df["manual_score"] = manual_scores_strict

valid_strict_df = strict_test_results_df[strict_test_results_df["is_valid_sample"] == True]

strict_test_score = valid_strict_df["manual_score"].mean()

print("Strict Prompt RAG Test Score:", strict_test_score)
print("Valid sample count:", len(valid_strict_df))

Strict Prompt RAG Test Score: 0.2631578947368421
Valid sample count: 19


In [44]:
strict_test_results_df.to_csv(
    f"{project_path}/outputs/metrics/strict_prompt_rag_testset_scored.csv",
    index=False,
    encoding="utf-8-sig"
)

pd.DataFrame([{
    "method": "Strict Prompt RAG - Hybrid Retrieval + Base LLM",
    "manual_accuracy": strict_test_score,
    "valid_sample_count": len(valid_strict_df),
    "total_sample_count": len(strict_test_results_df)
}]).to_csv(
    f"{project_path}/outputs/metrics/strict_prompt_rag_testset_score.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Strict prompt test score saved.")

Strict prompt test score saved.


In [45]:
base_test_results = []

for _, row in test_eval_df.iterrows():
    question = row["question"]
    expected_answer = row["answer"]

    retrieved = hybrid_retrieve_top_k_filtered(
        question,
        embedding_model,
        index,
        chunks_df,
        bm25,
        k=5,
        alpha=0.5
    )

    contexts = [r["chunk_text"] for r in retrieved[:3]]

    # 👇 BASE PROMPT (eski)
    prompt = build_rag_prompt(question, contexts)

    generated_answer = generate_answer(prompt)

    base_test_results.append({
        "question": question,
        "expected_answer": expected_answer,
        "generated_answer": generated_answer
    })

base_test_results_df = pd.DataFrame(base_test_results)

In [46]:
base_test_results_df["clean_generated_answer"] = base_test_results_df["generated_answer"].apply(clean_generated_answer)

In [47]:
for i, row in base_test_results_df.iterrows():
    print("=" * 120)
    print("INDEX:", i)
    print("QUESTION:")
    print(row["question"])
    print("\nEXPECTED:")
    print(row["expected_answer"])
    print("\nCLEAN GENERATED:")
    print(row["clean_generated_answer"][:500])

INDEX: 0
QUESTION:
Anayasanın 101. Maddesiyle İlgili Tartışmalar Nelerdir?

EXPECTED:
Cumhurbaşkanının seçilme şartlarının sınırları ve uygulanması üzerine tartışmalar olabilir.

CLEAN GENERATED:
Anayasanın 101. Madde'si, Uyuşmazlık Mahkemesi'nin görevi olan adli ve idari yargı mercileri arasındaki görev ve hüküm uyuşmazlıklarını kesin olarak çözümlemeye yetkilidir. Bu konuda tartışılabilen temel sorunlar, Uyuşmazlık Mahkemesi'nin görevleri, hüküm verme yetkisini taşıyan kişinin hangi kurumda bulunur, ve bu yetkisini kullanan kişinin ne tür hüküm
INDEX: 1
QUESTION:
Bir grup vatandaş, belirli bir etnik grubun diğerlerinden daha fazla hakka sahip olması için imza kampanyası başlatmıştır. Bu durum Anayasanın 10. Maddesi ile nasıl çelişir?

EXPECTED:
Anayasanın 10. Maddesi, herkesin kanun önünde eşit olduğunu belirtir. Bu tür bir imza kampanyası Anayasa'ya aykırıdır.

CLEAN GENERATED:
Bu durum Anayasanın 150. Madde ile ilgili olabilir. Bu madde, Anayasanın 10. Maddesine karşı, Cumhurbaşkan

In [48]:
base_test_results_df["is_valid_sample"] = True
base_test_results_df.loc[19, "is_valid_sample"] = False

manual_scores_base = [
    0.0,  # 0 yanlış
    0.0,  # 1 yanlış
    0.5,  # 2 kısmen
    0.0,  # 3 cevap bulunamadı
    0.5,  # 4 kısmen: kesin hüküm vermiyor, bilirkişi kısmı yok
    0.0,  # 5 cevap yok
    0.0,  # 6 yanlış
    0.0,  # 7 cevap bulunamadı
    0.0,  # 8 yanlış
    0.5,  # 9 kısmen
    0.0,  # 10 cevap yok
    0.0,  # 11 yanlış
    0.0,  # 12 yanlış
    1.0,  # 13 doğru
    0.5,  # 14 kısmen
    0.5,  # 15 kısmen
    0.5,  # 16 kısmen
    0.0,  # 17 cevap yok
    1.0,  # 18 doğru/kabul edilebilir
    0.0   # 19 invalid, skora katılmayacak
]

base_test_results_df["manual_score"] = manual_scores_base

valid_base_df = base_test_results_df[base_test_results_df["is_valid_sample"] == True]
base_test_score = valid_base_df["manual_score"].mean()

print("Base RAG Test Score:", base_test_score)
print("Valid sample count:", len(valid_base_df))

Base RAG Test Score: 0.2631578947368421
Valid sample count: 19


In [49]:
test_comparison_df = pd.DataFrame([
    {"method": "Base RAG", "manual_accuracy": base_test_score},
    {"method": "Strict Prompt RAG", "manual_accuracy": strict_test_score}
])

test_comparison_df

,method,manual_accuracy
0,Base RAG,0.263158
1,Strict Prompt RAG,0.263158
